Heavily stolen code from our friend Josh so I have a basic working transformer to start with:
https://github.com/StatQuest/decoder_transformer_from_scratch/blob/main/decoder_transformers_with_pytorch_and_lightning_v2.ipynb

In [1]:
import pip
try:
  __import__("lightning")
except ImportError:
  pip.main(['install', "lightning"])

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

import lightning as L

import os
import re

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


Collecting lightning

Downloading lightning-2.5.5-py3-none-any.whl.metadata (39 kB)

Requirement already satisfied: PyYAML<8.0,>5.4 in /usr/local/lib/python3.12/dist-packages (from lightning) (6.0.3)

Requirement already satisfied: fsspec<2027.0,>=2022.5.0 in /usr/local/lib/python3.12/dist-packages (from fsspec[http]<2027.0,>=2022.5.0->lightning) (2025.3.0)

...

Downloading pytorch_lightning-2.5.5-py3-none-any.whl (832 kB)

Output()

Installing collected packages: lightning-utilities, torchmetrics, pytorch-lightning, lightning

Successfully installed lightning-2.5.5 lightning-utilities-0.15.2 pytorch-lightning-2.5.5 torchmetrics-1.8.2

NumExpr defaulting to 2 threads.

In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

I love using Dracula.txt

In [3]:
destination_path = os.path.join(os.getcwd(), "dracula.txt")
torch.hub.download_url_to_file('https://www.gutenberg.org/cache/epub/345/pg345.txt', destination_path)

100%|██████████| 870k/870k [00:00<00:00, 7.28MB/s]


Contort text into input for the transformer

In [4]:
filepath = "dracula.txt"
seq_len = 8
dataSize = 5000 #training size

#load
with open(filepath, "r", encoding="utf-8") as f:
    text = f.read().lower()
    text = re.sub(r"[^a-z\s]", "", text)

tokens = text.split()

#build vocab
unique_tokens = sorted(set(tokens))
token_to_id = {tok: i for i, tok in enumerate(unique_tokens)}
token_to_id["<EOS>"] = len(token_to_id)
token_to_id["<UNK>"] = len(token_to_id)
id_to_token = {i: tok for tok, i in token_to_id.items()}

#convert to IDs
ids = [token_to_id.get(tok, token_to_id["<UNK>"]) for tok in tokens]

#dataset
inputs_list = []
labels_list = []

for i in range(len(ids) - seq_len - 1):
    seq = ids[i:i+seq_len]

    inp = seq + [token_to_id["<EOS>"]] #input gets EOS at end
    inp[-1], inp[-2] = inp[-2], inp[-1]
    lbl = inp[1:] + [token_to_id["<EOS>"]] #labels shift left + EOS

    inputs_list.append(inp)
    labels_list.append(lbl)

inputs = torch.tensor(inputs_list[:dataSize]).to(device)
labels = torch.tensor(labels_list[:dataSize]).to(device)


dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

print("Example input IDs:", inputs[0])
print("Decoded:", [id_to_token[i.item()] for i in inputs[4000]])
print("Decoded:", [id_to_token[i.item()] for i in labels[4000]])


Example input IDs: tensor([ 9380,  7165,  4120,  2877,  6376,  2719,  9436, 10706,  2877])
Decoded: ['excitement', 'kept', 'on', 'for', 'some', 'little', 'time', '<EOS>', 'and']
Decoded: ['kept', 'on', 'for', 'some', 'little', 'time', '<EOS>', 'and', '<EOS>']


Basic transformer construction without context expansion.

In [5]:
class PositionEncoding(nn.Module):

    def __init__(self, d_model=2, max_len=6):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1)
        embedding_index = torch.arange(start=0, end=d_model, step=2).float()

        div_term = 1/torch.tensor(10000.0)**(embedding_index / d_model)


        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)


    def forward(self, word_embeddings):

        return word_embeddings + self.pe[:word_embeddings.size(0), :]

In [6]:
class Attention(nn.Module):

    def __init__(self, d_model=2):

        super().__init__()

        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        self.row_dim = 0
        self.col_dim = 1


    def forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask=None):

        q = self.W_q(encodings_for_q)
        k = self.W_k(encodings_for_k)
        v = self.W_v(encodings_for_v)

        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))

        scaled_sims = sims / torch.tensor(k.size(self.col_dim)**0.5)

        if mask is not None:
            scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)

        attention_percents = F.softmax(scaled_sims, dim=self.col_dim)
        attention_scores = torch.matmul(attention_percents, v)

        return attention_scores

In [13]:
class DecoderOnlyTransformer(L.LightningModule):

    def __init__(self, num_tokens=4, d_model=2, max_len=6):

        super().__init__()

        L.seed_everything(seed=42)

        self.we = nn.Embedding(num_embeddings=num_tokens,
                               embedding_dim=d_model)
        self.pe = PositionEncoding(d_model=d_model,
                                   max_len=max_len)
        self.self_attention = Attention(d_model=d_model)
        self.fc_layer = nn.Linear(in_features=d_model, out_features=num_tokens)

        self.loss = nn.CrossEntropyLoss()


    def forward(self, token_ids):

        word_embeddings = self.we(token_ids)
        position_encoded = self.pe(word_embeddings)

        mask = torch.tril(torch.ones((token_ids.size(dim=0), token_ids.size(dim=0)))).to(device)
        mask = mask == 0

        self_attention_values = self.self_attention(position_encoded,
                                                    position_encoded,
                                                    position_encoded,
                                                    mask=mask)

        residual_connection_values = position_encoded + self_attention_values
        fc_layer_output = self.fc_layer(residual_connection_values)

        return fc_layer_output


    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)


    def training_step(self, batch, batch_idx):
        input_tokens, labels = batch # collect input
        output = self.forward(input_tokens[0])
        loss = self.loss(output, labels[0])
        #self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)

        return loss

In [35]:
model = DecoderOnlyTransformer(num_tokens=len(token_to_id), d_model=100, max_len=9)
model.to(device)

INFO: Seed set to 42


Seed set to 42

DecoderOnlyTransformer(
  (we): Embedding(10708, 100)
  (pe): PositionEncoding()
  (self_attention): Attention(
    (W_q): Linear(in_features=100, out_features=100, bias=False)
    (W_k): Linear(in_features=100, out_features=100, bias=False)
    (W_v): Linear(in_features=100, out_features=100, bias=False)
  )
  (fc_layer): Linear(in_features=100, out_features=10708, bias=True)
  (loss): CrossEntropyLoss()
)

In [39]:
trainer = L.Trainer(max_epochs=3)
trainer.fit(model, train_dataloaders=dataloader)

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

INFO: GPU available: False, used: False


GPU available: False, used: False

INFO: TPU available: False, using: 0 TPU cores


...

INFO: 
  | Name           | Type             | Params | Mode 
------------------------------------------------------------
0 | we             | Embedding        | 1.1 M  | train
1 | pe             | PositionEncoding | 0      | train
2 | self_attention | Attention        | 30.0 K | train
3 | fc_layer       | Linear           | 1.1 M  | train
4 | loss           | CrossEntropyLoss | 0      | train
------------------------------------------------------------
2.2 M     Trainable params
0         Non-trainable params
2.2 M     Total params
8.729     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode


| Name           | Type             | Params | Mode 
------------------------------------------------------------
0 | we             | Embedding        | 1.1 M  | train
1 | pe             | PositionEncoding | 0      | train
2 | self_attention | Attention        | 30.0 K | train
3 | fc_layer       | Linear           | 1.1 M  | train
4 | loss           | CrossEntropyLoss | 0      | train
------------------------------------------------------------
2.2 M     Trainable params
0         Non-trainable params
2.2 M     Total params
8.729     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode

Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=3` reached.


`Trainer.fit` stopped: `max_epochs=3` reached.

In [42]:
max_length = 6
model_input = torch.tensor([token_to_id["for"],
                            token_to_id["some"],
                            token_to_id["little"],
                            token_to_id["<EOS>"]])
input_length = model_input.size(dim=0)

predictions = model(model_input)
predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
predicted_ids = predicted_id

for i in range(input_length, max_length):
    if (predicted_id == token_to_id["<EOS>"]): # if the prediction is <EOS>, then we are done
        break

    model_input = torch.cat((model_input, predicted_id))

    predictions = model(model_input)
    predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
    predicted_ids = torch.cat((predicted_ids, predicted_id))

print("Predicted Tokens:\n")
for id in predicted_ids:
    print("\t", id_to_token[id.item()])

Predicted Tokens:

	 increased
	 experiences
	 waited


Looks like it works to some degree.

Let's add something like sparse attention.

In [43]:
class EXPositionEncoding(nn.Module):

    def __init__(self, d_model=2, max_len=6):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1)
        embedding_index = torch.arange(start=0, end=d_model, step=2).float()

        div_term = 1/torch.tensor(10000.0)**(embedding_index / d_model)


        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe)


    def forward(self, word_embeddings):

        return word_embeddings + self.pe[:word_embeddings.size(0), :]

Attempt at adding some sort of sparse mask to limit computation.

In [66]:
class EXAttention(nn.Module):

    def __init__(self, d_model=2):

        super().__init__()

        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        self.row_dim = 0
        self.col_dim = 1


    def forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask=None, window=3):
      q = self.W_q(encodings_for_q)
      k = self.W_k(encodings_for_k)
      v = self.W_v(encodings_for_v)

      sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
      scaled_sims = sims / (k.size(self.col_dim) ** 0.5)

      seq_len, _ = scaled_sims.shape

      block_mask = torch.ones_like(scaled_sims, dtype=torch.bool)

      for i in range(seq_len):
          #allow attention within window around token i
          start = max(0, i - window)
          end = min(seq_len, i + window + 1)
          block_mask[..., i, start:end] = False

      if mask is not None:
          final_mask = torch.logical_or(mask, block_mask)
      else:
          final_mask = block_mask

      scaled_sims = scaled_sims.masked_fill(final_mask, -1e9)

      attention_percents = F.softmax(scaled_sims, dim=self.col_dim)
      attention_scores = torch.matmul(attention_percents, v)

      return attention_scores

In [47]:
class EXDecoderOnlyTransformer(L.LightningModule):

    def __init__(self, num_tokens=4, d_model=2, max_len=6):

        super().__init__()

        L.seed_everything(seed=42)

        self.we = nn.Embedding(num_embeddings=num_tokens,
                               embedding_dim=d_model)
        self.pe = EXPositionEncoding(d_model=d_model,
                                   max_len=max_len)
        self.self_attention = EXAttention(d_model=d_model)
        self.fc_layer = nn.Linear(in_features=d_model, out_features=num_tokens)

        self.loss = nn.CrossEntropyLoss()


    def forward(self, token_ids):

        word_embeddings = self.we(token_ids)
        position_encoded = self.pe(word_embeddings)

        mask = torch.tril(torch.ones((token_ids.size(dim=0), token_ids.size(dim=0)))).to(device)
        mask = mask == 0

        self_attention_values = self.self_attention(position_encoded,
                                                    position_encoded,
                                                    position_encoded,
                                                    mask=mask)

        residual_connection_values = position_encoded + self_attention_values
        fc_layer_output = self.fc_layer(residual_connection_values)

        return fc_layer_output


    def configure_optimizers(self):
        return Adam(self.parameters(), lr=0.1)


    def training_step(self, batch, batch_idx):
        input_tokens, labels = batch # collect input
        output = self.forward(input_tokens[0])
        loss = self.loss(output, labels[0])

        return loss

In [67]:
EXmodel = EXDecoderOnlyTransformer(num_tokens=len(token_to_id), d_model=100, max_len=9)
EXmodel.to(device)

INFO: Seed set to 42


Seed set to 42

EXDecoderOnlyTransformer(
  (we): Embedding(10708, 100)
  (pe): EXPositionEncoding()
  (self_attention): EXAttention(
    (W_q): Linear(in_features=100, out_features=100, bias=False)
    (W_k): Linear(in_features=100, out_features=100, bias=False)
    (W_v): Linear(in_features=100, out_features=100, bias=False)
  )
  (fc_layer): Linear(in_features=100, out_features=10708, bias=True)
  (loss): CrossEntropyLoss()
)

In [68]:
EXtrainer = L.Trainer(max_epochs=3)
EXtrainer.fit(EXmodel, train_dataloaders=dataloader)

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

INFO: GPU available: False, used: False


GPU available: False, used: False

INFO: TPU available: False, using: 0 TPU cores


...

INFO: 
  | Name           | Type               | Params | Mode 
--------------------------------------------------------------
0 | we             | Embedding          | 1.1 M  | train
1 | pe             | EXPositionEncoding | 0      | train
2 | self_attention | EXAttention        | 30.0 K | train
3 | fc_layer       | Linear             | 1.1 M  | train
4 | loss           | CrossEntropyLoss   | 0      | train
--------------------------------------------------------------
2.2 M     Trainable params
0         Non-trainable params
2.2 M     Total params
8.729     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode


| Name           | Type               | Params | Mode 
--------------------------------------------------------------
0 | we             | Embedding          | 1.1 M  | train
1 | pe             | EXPositionEncoding | 0      | train
2 | self_attention | EXAttention        | 30.0 K | train
3 | fc_layer       | Linear             | 1.1 M  | train
4 | loss           | CrossEntropyLoss   | 0      | train
--------------------------------------------------------------
2.2 M     Trainable params
0         Non-trainable params
2.2 M     Total params
8.729     Total estimated model params size (MB)
8         Modules in train mode
0         Modules in eval mode

Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=3` reached.


`Trainer.fit` stopped: `max_epochs=3` reached.

In [70]:
max_length = 6
model_input = torch.tensor([token_to_id["for"],
                            token_to_id["some"],
                            token_to_id["little"],
                            token_to_id["<EOS>"]])
input_length = model_input.size(dim=0)

predictions = EXmodel(model_input)
predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
predicted_ids = predicted_id

for i in range(input_length, max_length):
    if (predicted_id == token_to_id["<EOS>"]): # if the prediction is <EOS>, then we are done
        break

    model_input = torch.cat((model_input, predicted_id))

    predictions = EXmodel(model_input)
    predicted_id = torch.tensor([torch.argmax(predictions[-1,:])])
    predicted_ids = torch.cat((predicted_ids, predicted_id))

print("Predicted Tokens:\n")
for id in predicted_ids:
    print("\t", id_to_token[id.item()])

Predicted Tokens:

	 salient
	 rear
	 driver


Also looks like it works to some degree.

Both trained in about 12 minutes, so not really any speedup or slowdown. Since we're working at such a small scale and not with super long context windows, it's not really possible to see if our sparse attention actually does what it is supposed to. I'd go for much longer context windows if it didn't already take several minutes to train with 6 length sequences, however, it is now my bedtime. My main metric of success is if the modified version still appears to work and not completely explode.

Thank you Josh and Squatch!